# Честная валидация, подбор гиперпараметров и регуляризация

## Содержание

0. Подготовка среды и подключение Weights & Biases
1. Данные: цены на жильё в Калифорнии
2. Одиночное разбиение и его проблема
3. Кросс-валидация
4. Подбор гиперпараметров
5. Регуляризация: Ridge, Lasso, ElasticNet
6. Практикум

Это занятие продолжает второе: там вы строили линейную регрессию на своём датасете и обещали вернуться к регуляризации (Ridge) из P.S. Теперь разбираем её подробно — но сначала про то, **как честно измерять качество**: почему одной отложенной выборки мало и как правильно подбирать параметры модели.

Демонстрационный пример — предсказание цены дома (датасет California Housing). Практикум (раздел 6) вы выполняете на **датасете своего проекта** — том же, что в занятиях 2–3. Запуски логируем в Weights & Biases. Выводы после каждого блока записывайте **в ячейке markdown прямо в блокноте**.

## 0. Подготовка среды и подключение Weights & Biases

К библиотекам прошлых занятий ничего нового не добавляется. **Примечание.** Если с W&B проблемы, замените `wandb.init(...)` на `wandb.init(mode="disabled")` — код продолжит выполняться.

In [ ]:
import os
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import wandb

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (8, 5)
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
wandb.login()

## 1. Данные: цены на жильё в Калифорнии

Классический датасет `housing.csv` (20 640 районов Калифорнии). Целевая переменная — медианная стоимость дома в районе, есть числовые признаки, один категориальный и немного пропусков. Идеально подходит для честного `Pipeline`.

**Примечание.** Целевая переменная «обрезана» сверху значением 500 001 — это артефакт исходных данных, о котором стоит помнить при интерпретации ошибок.

In [ ]:
URL = ("https://raw.githubusercontent.com/ageron/handson-ml2/"
       "master/datasets/housing/housing.csv")
LOCAL_PATH = "housing.csv"

if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
else:
    df = pd.read_csv(URL)
    df.to_csv(LOCAL_PATH, index=False)

print("Размер таблицы:", df.shape)
print("Пропуски:", int(df.isna().sum().sum()), "| max целевой:", df['median_house_value'].max())
df.head()

In [ ]:
num = ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income']
cat = ['ocean_proximity']

X = df[num + cat]
y = df['median_house_value']

# test откладываем сразу и больше не трогаем — на нём только финальная оценка
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print("Train / test:", len(X_train), len(X_test))


def make_model(reg, degree=1):
    """Pipeline: пропуски -> масштаб -> полиномиальные признаки -> регрессор.

    Целевую переменную стандартизуем (TransformedTargetRegressor), чтобы сила
    регуляризации alpha не зависела от порядка цен в тысячах долларов.
    """
    num_block = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc', StandardScaler()),
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
    ])
    prep = ColumnTransformer([
        ('num', num_block, num),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat),
    ])
    return TransformedTargetRegressor(
        regressor=Pipeline([('prep', prep), ('model', reg)]),
        transformer=StandardScaler(),
    )


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

## 2. Одиночное разбиение и его проблема

### Математический аппарат

Оценка ошибки на одной отложенной выборке — это **случайная величина**: при другом разбиении (другом `random_state`) она будет другой. Её дисперсия зависит от размера выборки и от «капризности» данных:

$$\widehat{RMSE} = \sqrt{\frac{1}{n_{val}}\sum_{i \in val}(y_i - \hat{y}_i)^2}$$

*Смысл:* одна цифра ничего не говорит о том, насколько она стабильна. Прежде чем сравнивать модели, полезно понять **разброс** оценки.

**Задача 1.**
Оцените одну и ту же модель на восьми разных разбиениях (меняя `random_state`) и посмотрите на разброс RMSE. Объясните, почему одной отложенной выборки для выводов недостаточно. Ответ — в «Выводе».

In [ ]:
scores = []
for seed in range(8):
    Xa, Xb, ya, yb = train_test_split(X_train, y_train, test_size=0.25, random_state=seed)
    m = make_model(LinearRegression(), degree=1).fit(Xa, ya)
    scores.append(rmse(yb, m.predict(Xb)))

print("RMSE по 8 разным разбиениям:")
print([round(s) for s in scores])
print(f"min={min(scores):.0f}  max={max(scores):.0f}  "
      f"mean={np.mean(scores):.0f}  std={np.std(scores):.0f}")

**Вывод:**

_..._

**Примечание.** Есть и вторая ловушка — **утечка (leakage)**. Если обучить `StandardScaler`, `SimpleImputer` или `PolynomialFeatures` на **всём** датасете до разбиения, их статистики «подсмотрят» на validation и оценка окажется завышенной. Поэтому весь препроцессинг держим внутри `Pipeline`.

## 3. Кросс-валидация

### Математический аппарат

K-fold делит данные на $K$ частей: модель обучается на $K-1$ и проверяется на оставшейся, и так $K$ раз. Итоговая оценка — среднее по фолдам:

$$\widehat{RMSE} = \frac{1}{K}\sum_{k=1}^{K} RMSE_k$$

*Смысл:* каждый объект успевает побыть и в обучении, и в проверке. Оценка использует все данные и усредняет случайность разбиения, поэтому у неё **меньше дисперсия**, чем у одного hold-out. Разброс по фолдам ($std$) показывает стабильность оценки.

Схемы выбирают под задачу: `KFold` (регрессия), `StratifiedKFold` (классификация — сохраняет доли классов), `GroupKFold` (сгруппированные объекты), `TimeSeriesSplit` (временные ряды — нельзя перемешивать).

**Задача 2.**
Сравните разброс одиночных разбиений из раздела 2 и разброс по фолдам кросс-валидации. Почему усреднение по фолдам даёт более надёжную оценку? Ответ — в «Выводе».

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scores = cross_val_score(
    make_model(LinearRegression(), degree=1),
    X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error',
)
scores = -scores
print("RMSE по фолдам:", [round(s) for s in scores])
print(f"CV RMSE = {scores.mean():.0f} ± {scores.std():.0f}")

# Задача 2: ваша ячейка для расчётов и сравнения


**Вывод:**

_..._

## 4. Подбор гиперпараметров

### Математический аппарат

`GridSearchCV` перебирает сетку гиперпараметров и выбирает тот, у которого лучше среднее качество по кросс-валидации. `RandomizedSearchCV` делает то же по случайным точкам (быстрее при больших сетках), а **вложенная** кросс-валидация отделяет подбор гиперпараметров от финальной оценки.

*Смысл:* гиперпараметры — это часть модели, и подбирать их нужно на **train** (через CV), а не на test. Иначе оценка на test перестаёт быть честной: вы подгоняете решение под него.

Сначала посмотрим, как сложность модели (степень полинома) влияет на переобучение.

In [ ]:
for degree in [1, 2, 3]:
    m = make_model(LinearRegression(), degree=degree).fit(X_train, y_train)
    print(f"degree={degree}: train RMSE={rmse(y_train, m.predict(X_train)):.0f}  "
          f"test RMSE={rmse(y_test, m.predict(X_test)):.0f}")

**Примечание.** При `degree=3` модель почти идеально подгоняет train, но на отложенных данных заметно хуже — это переобучение. Спасает регуляризация, но сначала подберём её силу честно.

In [ ]:
alphas = np.logspace(-3, 3, 13)
rows = []
for a in alphas:
    s = -cross_val_score(make_model(Ridge(alpha=a), degree=3),
                         X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error')
    rows.append((a, s.mean(), s.std()))

alpha_table = pd.DataFrame(rows, columns=['alpha', 'CV_RMSE', 'CV_std'])
print(alpha_table.to_string(index=False, formatters={
    'alpha': lambda x: f"{x:g}", 'CV_RMSE': lambda x: f"{x:,.0f}", 'CV_std': lambda x: f"{x:,.0f}"}))

plt.figure(figsize=(8, 5))
plt.errorbar(alpha_table['alpha'], alpha_table['CV_RMSE'], yerr=alpha_table['CV_std'], marker='o', capsize=3)
plt.xscale('log')
plt.xlabel('alpha (сила регуляризации)')
plt.ylabel('CV RMSE')
plt.title('Ridge: качество против силы регуляризации')
plt.show()

In [ ]:
gs = GridSearchCV(
    make_model(Ridge(), degree=3),
    {'regressor__model__alpha': np.logspace(-3, 3, 13)},
    cv=cv, scoring='neg_root_mean_squared_error', n_jobs=-1,
)
gs.fit(X_train, y_train)

best_alpha = gs.best_params_['regressor__model__alpha']
best_model = gs.best_estimator_
print(f"Лучший alpha по кросс-валидации: {best_alpha:.1f}")
print(f"CV RMSE лучшей модели: {-gs.best_score_:.0f}")

p_test = best_model.predict(X_test)
print("\nФинальная оценка на test (один раз):")
print(f"  RMSE = {rmse(y_test, p_test):.0f}")
print(f"  MAE  = {mean_absolute_error(y_test, p_test):.0f}")
print(f"  R2   = {r2_score(y_test, p_test):.3f}")

run = wandb.init(project="apml-course", name="cv-ridge-housing",
                 config={"model": "Ridge", "poly_degree": 3, "alpha": float(best_alpha)})
wandb.log({"test_rmse": rmse(y_test, p_test), "test_r2": r2_score(y_test, p_test)})
run.finish()

**Задача 3.**
Посмотрите на график CV RMSE от `alpha`. Объясните его форму через bias–variance: почему при очень малом `alpha` качество хуже (особенно при degree 3), а при очень большом — тоже плохо? Ответ — в «Выводе».

**Вывод:**

_..._

## 5. Регуляризация: Ridge, Lasso, ElasticNet

### Математический аппарат

Регуляризация добавляет к функции потерь штраф за величину коэффициентов:

$$L_{Ridge} = L + \lambda\|w\|_2^2, \qquad L_{Lasso} = L + \lambda\|w\|_1, \qquad L_{EN} = L + \lambda\big(\rho\|w\|_1 + \tfrac{1-\rho}{2}\|w\|_2^2\big)$$

- **Ridge (L2)** плавно сжимает все коэффициенты, снижая дисперсию модели;
- **Lasso (L1)** умеет **обнулять** коэффициенты — встроенный отбор признаков;
- **ElasticNet** сочетает оба эффекта.

У Ridge при центрированных признаках есть явное решение:

$$\hat{w} = (X^\top X + \lambda I)^{-1} X^\top y$$

*Смысл:* добавка $\lambda I$ делает матрицу обратимой даже при мультиколлинеарности (проблема из P.S. второго занятия) — поэтому Ridge стабилизирует решение.

**Интерактив.** Прежде чем считать, поиграйте с регуляризацией руками: меняйте тип (L1/L2), силу $\lambda$, уровень шума и число признаков — https://www.interactive-ml.com/regularization.html. Обратите внимание, как L1 обнуляет часть весов, а L2 только сжимает.

In [ ]:
# Lasso обнуляет коэффициенты при росте alpha
rows = []
for a in [0.001, 0.01, 0.1, 1.0]:
    m = make_model(Lasso(alpha=a, max_iter=10000), degree=2).fit(X_train, y_train)
    coef = m.regressor_.named_steps['model'].coef_
    rows.append((a, int(np.sum(np.abs(coef) > 1e-8)), len(coef),
                 rmse(y_train, m.predict(X_train)), rmse(y_test, m.predict(X_test))))

sparsity = pd.DataFrame(rows, columns=['alpha', 'nonzero', 'total', 'train_RMSE', 'test_RMSE'])
print(sparsity.to_string(index=False))

**Примечание.** Ridge только сжимает коэффициенты, Lasso обнуляет часть из них, а ElasticNet сглаживает поведение Lasso при коррелирующих признаках (за это отвечает параметр `l1_ratio`). Сравним все четыре модели на одной кросс-валидации (берём `degree=2`: при `degree=3` Lasso и ElasticNet считаются заметно дольше).

In [ ]:
linear_models = {
    'OLS (без регуляризации)': LinearRegression(),
    'Ridge (L2)': Ridge(alpha=10.0),
    'Lasso (L1)': Lasso(alpha=0.01, max_iter=10000),
    'ElasticNet (L1+L2)': ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000),
}
rows = []
for name, reg in linear_models.items():
    s = -cross_val_score(make_model(reg, degree=2), X_train, y_train, cv=cv,
                         scoring='neg_root_mean_squared_error')
    rows.append((name, s.mean(), s.std()))

linear_table = pd.DataFrame(rows, columns=['model', 'CV_RMSE', 'CV_std'])
print(linear_table.to_string(index=False, formatters={
    'CV_RMSE': lambda x: f"{x:,.0f}", 'CV_std': lambda x: f"{x:,.0f}"}))

**Задача 4.**
Почему регуляризация дала большой выигрыш при `degree=3`, но почти не помогает на линейной модели (`degree=1`)? От чего вообще зависит её польза (объём данных, число признаков, мультиколлинеарность)? Ответ — в «Выводе».

**Вывод:**

_..._

**Задача 5 (на вывод).**
Получите формулу Ridge-оценки $\hat{w} = (X^\top X + \lambda I)^{-1} X^\top y$: приравняйте градиент $\|Xw - y\|^2 + \lambda\|w\|^2$ к нулю и выразите $w$. Объясните, почему при мультиколлинеарности слагаемое $\lambda I$ спасает ситуацию.

**Ваш вывод:**

_..._

## 6. Практикум

Все задания — на **датасете своего проекта** (том же, что в занятиях 2–3). Каждый запуск логируйте в W&B отдельным run. После каждого задания заполните ячейку **«Вывод»**.

Если ваша задача — классификация, вместо Ridge/Lasso/ElasticNet используйте логистическую регрессию с `penalty='l2'`/`'l1'`/`'elasticnet'` (смысл тот же, меняется только функция потерь).

### Задание 6.1. Постановка задачи

Опишите словами: какую величину вы предсказываете и в каких единицах; какой метрикой измеряете качество и почему именно ей; как поймёте, что модель «достаточно хорошая».

**Ваш ответ:**

_..._

### Задание 6.2. Честная оценка

Соберите `Pipeline` (препроцессинг + модель) для своей задачи и оцените качество двумя способами: на одном отложенном разбиении и через кросс-валидацию. Сравните полученные цифры и их разброс — какой оценке вы доверяете больше и почему?

**Вывод:**

_..._

### Задание 6.3. Регуляризация: Ridge, Lasso, ElasticNet

Обучите три регуляризованные модели — Ridge, Lasso и ElasticNet (для классификации — логистическую регрессию с `penalty='l2'`, `'l1'` и `'elasticnet'`). Для каждой подберите силу регуляризации **по кросс-валидации на train** (`GridSearchCV`).

**Вывод:**

_..._

### Задание 6.4. Сравнение моделей

Сведите в одну таблицу все модели — нерегуляризованную базовую (OLS для регрессии, логистическую регрессию без штрафа для классификации) и три регуляризованные — на одной кросс-валидации. Выберите лучшую и один раз оцените её на test. Что дала регуляризация: улучшила качество или хотя бы стабилизировала оценку (уменьшила разброс)? Если почти ничего не изменилось — объясните, от чего это зависит (объём данных, число признаков, мультиколлинеарность). Для Lasso посмотрите, сколько коэффициентов обнулилось и совпадает ли это с вашим представлением о важных признаках.

**Вывод:**

_..._

### Задание 6.5. Итоговое резюме

В 5–7 предложениях: какой пайплайн (препроцессинг + модель + гиперпараметры) вы зафиксируете; какую метрику считаете главной; что показала регуляризация; что из этого войдёт в итоговый проект на зачёт.

**Резюме:**

_..._

---

## P.S. Кривые валидации и обучения

Два диагностических инструмента помогают понять, **куда двигаться**, когда качество не устраивает.

**Кривая валидации** (`validation_curve`) показывает качество на train и validation в зависимости от одного гиперпараметра:

- обе кривые низкие и близкие → недообучение (модель слишком простая);
- train высокий, validation низкий, большой разрыв → переобучение (усильте регуляризацию или упростите модель);
- «клюшка» в середине → есть оптимальный баланс.

**Кривая обучения** (`learning_curve`) показывает качество в зависимости от **объёма обучающей выборки**:

- кривые ещё сходятся в правом краю → поможет больше данных;
- вышли на плато с разрывом → данные не спасут, нужны признаки или другая модель.

*Смысл:* прежде чем «крутить» модель вслепую, посмотрите на кривые и решите, чего не хватает.

### Справочные материалы

- Интерактивная регуляризация (L1/L2): https://www.interactive-ml.com/regularization.html
- Кросс-валидация: https://scikit-learn.org/stable/modules/cross_validation.html
- Подбор гиперпараметров: https://scikit-learn.org/stable/modules/grid_search.html
- Полиномиальные признаки: https://scikit-learn.org/stable/modules/preprocessing.html#polynomial-features
- Линейные модели и регуляризация (Ridge/Lasso/ElasticNet): https://scikit-learn.org/stable/modules/linear_model.html
- Кривые валидации и обучения: https://scikit-learn.org/stable/modules/learning_curve.html